In [39]:
import pandas as pd
import sqlite3


pd.set_option('display.max_rows', None)

## Load data from the files

In [40]:

# Load your cleaned weather data
weather_df = pd.read_csv("../cleaned_climate_data.csv")

# Load the SimpleMaps cities dataset
simple_map_cities_df = pd.read_csv("../worldcities.csv")

## General information about the datasets

In [41]:
weather_df.head()

,city,region,country,month,high_temp_F,low_temp_F,mean_temp_F,precipitation_in,humidity_percent,dew_point_F,wind_mph,pressure_Hg,visibility_mi
0,Accra,NaN,Ghana,January,89.0,75.0,82.0,1.57,75.0,73.0,14.0,29.85,5.0
1,Accra,NaN,Ghana,February,90.0,77.0,84.0,1.79,77.0,75.0,17.0,29.84,6.0
2,Accra,NaN,Ghana,March,91.0,78.0,84.0,2.29,78.0,76.0,17.0,29.83,7.0
3,Accra,NaN,Ghana,April,90.0,78.0,84.0,2.93,78.0,76.0,16.0,29.83,7.0
4,Accra,NaN,Ghana,May,89.0,77.0,83.0,5.11,80.0,75.0,15.0,29.87,8.0


In [42]:
simple_map_cities_df.head()

,city,city_ascii,lat,lng,country,iso2,iso3,admin_name,capital,population,id
0,Tokyo,Tokyo,35.6850,139.7514,Japan,JP,JPN,Tōkyō,primary,39105000.0,1.392686e+09
1,Jakarta,Jakarta,-6.1753,106.8269,Indonesia,ID,IDN,Jakarta,primary,33756000.0,1.360771e+09
2,Guangzhou,Guangzhou,23.1300,113.2600,China,CN,CHN,Guangdong,admin,26940000.0,1.156237e+09
3,Mumbai,Mumbai,19.0758,72.8775,India,IN,IND,Mahārāshtra,admin,24973000.0,1.356227e+09
4,Shanghai,Shanghai,31.2325,121.4692,China,CN,CHN,Shanghai,admin,24870895.0,1.156074e+09


## Find unique countries in the simple map dataset

In [43]:
simple_map_countries = simple_map_cities_df['country'].unique()
print(f"Total number of contries in simple map df: {len(simple_map_countries)}")

Total number of contries in simple map df: 242


## Find unique countries in the weather dataset

In [44]:
weather_countries = weather_df['country'].unique()
print(f"Total number of contries in weather df: {len(weather_countries)}")

Total number of contries in weather df: 92


## Find differences between two datasets

In [45]:
diff = set(weather_countries).difference(set(simple_map_countries))
print(diff)

{'USA', 'Myanmar', 'Congo Dem. Rep.', 'Bahamas', 'South Korea'}


## Fix the mismatches between two datasets

In [47]:
# A list of keywords to search for in the SimpleMaps country column
search_terms = [
    'Congo', 
    'Bahama', 
    'Kazak', 
    'Czech', 
    'Kore', 
    'United',  # For USA and UK
    'Myanmar', 
    'Burma'    # Sometimes Myanmar is listed as Burma
]

print("--- SimpleMaps Country Matches ---")
for term in search_terms:
    # Find all unique country names that contain the search term
    matches = simple_map_cities_df[
        simple_map_cities_df['country'].str.contains(term, case=False, na=False)
    ]['country'].unique()
    
    print(f"Searching '{term}': {matches}")

--- SimpleMaps Country Matches ---
Searching 'Congo': ['Congo (Kinshasa)' 'Congo (Brazzaville)']
Searching 'Bahama': ['Bahamas, The']
Searching 'Kazak': ['Kazakhstan']
Searching 'Czech': ['Czechia']
Searching 'Kore': ['Korea, South' 'Korea, North']
Searching 'United': ['United States' 'United Kingdom' 'United Arab Emirates']
Searching 'Myanmar': []
Searching 'Burma': ['Burma']


In [48]:
country_corrections = {
    "Congo Dem. Rep.": "Congo (Kinshasa)",
    "Bahamas": "Bahamas, The",
    "Kazakstan": "Kazakhstan",
    "Czech Republic": "Czechia",
    "South Korea": "Korea, South",
    "USA": "United States",
    "UK": "United Kingdom",
    "Myanmar": "Burma"
}

# Apply the corrections to scraped weather data
weather_df['country'] = weather_df['country'].replace(country_corrections)

In [49]:
simple_map_cities_df = simple_map_cities_df.rename(columns={"admin_name": "region"})

In [50]:
def create_join_keys(df, columns):
    """
    Creates temporary '_clean' columns for merging by stripping accents, 
    whitespace.
    """
    # Work on a copy to avoid SettingWithCopy warnings
    df_clean = df.copy()
    
    for col in columns:
        clean_col_name = f"{col}_clean"
        df_clean[clean_col_name] = (
            df_clean[col]
            .str.normalize('NFKD')               # Decompose accented characters
            .str.encode('ascii', errors='ignore') # Drop the accents
            .str.decode('utf-8')                 # Convert back to standard string
            .str.strip()                         # Remove leading/trailing spaces
        )
    return df_clean

In [51]:
geo_columns = ['city', 'region', 'country']

# Generate the clean join keys for both datasets
weather_df = create_join_keys(weather_df, geo_columns)
simple_map_cities_df = create_join_keys(simple_map_cities_df, geo_columns)

In [52]:
simple_map_countries = simple_map_cities_df['country_clean'].unique()
print(f"Total number of contries in simple map df: {len(simple_map_countries)}")

weather_countries = weather_df['country_clean'].unique()
print(f"Total number of contries in weather df: {len(weather_countries)}")

diff = set(weather_countries).difference(set(simple_map_countries))
print(f"Countries in weather df but not in simple map df: {diff}")


Total number of contries in simple map df: 242
Total number of contries in weather df: 92
Countries in weather df but not in simple map df: set()


In [53]:
simple_map_cities = simple_map_cities_df['city_clean'].unique()
print(f"Total number of cities in simple map df: {len(simple_map_cities)}")

Total number of cities in simple map df: 46256


In [54]:
weather_cities = weather_df['city_clean'].unique()
print(f"Total number of cities in weather df: {len(weather_cities)}")

Total number of cities in weather df: 140


In [55]:
diff = set(weather_cities).difference(set(simple_map_cities))
print(diff)

{'Yangon', 'Kiritimati', 'Bengaluru', 'Washington DC'}


In [56]:
# A list of partial names to search for in the SimpleMaps city column
search_cities = [
    'Yangon', 'Rangoon',      # Checking both names for Myanmar's largest city
    'Kiritimati', 'Christmas',# Kiritimati is also known as Christmas Island
    'Bengaluru', 'Bangalore', # Checking both names for the Indian city
    'Washington'              # Checking Washington DC
]

print("--- SimpleMaps City Matches ---")
for term in search_cities:
    # Find all unique city names that contain the search term
    matches = simple_map_cities_df[
        simple_map_cities_df['city_clean'].str.contains(term, case=False, na=False)
    ]['city_clean'].unique()
    
    # We only print if it actually found a match to keep the output clean
    if len(matches) > 0:
        print(f"Searching '{term}': {matches}")

--- SimpleMaps City Matches ---
Searching 'Rangoon': ['Rangoon']
Searching 'Bangalore': ['Bangalore']
Searching 'Washington': ['Washington' 'New Washington' 'Fort Washington' 'Mount Washington'
 'Port Washington' 'Washington Court House' 'Washington Terrace']


In [57]:
city_corrections = {
    "Washington DC": "Washington",
    "Bengaluru": "Bangalore",
    "Yangon": "Rangoon" 
}

# Apply the corrections to your scraped weather data
weather_df['city_clean'] = weather_df['city_clean'].replace(city_corrections)

In [58]:
simple_map_cities = simple_map_cities_df['city_clean'].unique()
print(f"Total number of cities in simple map df: {len(simple_map_cities)}")

weather_cities = weather_df['city_clean'].unique()
print(f"Total number of cities in weather df: {len(weather_cities)}")

diff = set(weather_cities).difference(set(simple_map_cities))
print(f"Cities in weather df but not in simple map df: {diff}")

Total number of cities in simple map df: 46256
Total number of cities in weather df: 140
Cities in weather df but not in simple map df: {'Kiritimati'}


In [59]:
# Create sets of tuples (city, region, country) for both datasets
weather_set = set(weather_df[['city_clean', 'region_clean', 'country_clean']].itertuples(index=False, name=None))
map_set = set(simple_map_cities_df[['city_clean', 'region_clean', 'country_clean']].itertuples(index=False, name=None))

# Subtract the map set from the weather set to see what is leftover
missing_in_maps = weather_set - map_set
print(missing_in_maps)
print(f"Total number of missing entries in SimpleMaps: {len(missing_in_maps)}")

{('Kyiv', nan, 'Ukraine'), ('Johannesburg', nan, 'South Africa'), ('Zagreb', nan, 'Croatia'), ('Dhaka', nan, 'Bangladesh'), ('Addis Ababa', nan, 'Ethiopia'), ('Amman', nan, 'Jordan'), ('Tehran', nan, 'Iran'), ('Belgrade', nan, 'Serbia'), ('Kuala Lumpur', nan, 'Malaysia'), ('Shanghai', 'Shanghai Municipality', 'China'), ('Kiritimati', 'Christmas Island', 'Kiribati'), ('Bangkok', nan, 'Thailand'), ('Washington', nan, 'United States'), ('Reykjavik', nan, 'Iceland'), ('Dublin', nan, 'Ireland'), ('Riyadh', nan, 'Saudi Arabia'), ('Kinshasa', nan, 'Congo (Kinshasa)'), ('Seoul', nan, 'Korea, South'), ('Dubai', 'Dubai', 'United Arab Emirates'), ('Cape Town', nan, 'South Africa'), ('Santiago', nan, 'Chile'), ('Brussels', nan, 'Belgium'), ('Rome', nan, 'Italy'), ('Nairobi', nan, 'Kenya'), ('Manila', nan, 'Philippines'), ('Auckland', nan, 'New Zealand'), ('Minsk', nan, 'Belarus'), ('Lisbon', nan, 'Portugal'), ('San Juan', nan, 'Puerto Rico'), ('Barcelona', 'Barcelona', 'Spain'), ('Kuwait City', na

Since the scraped data focuses on the "most popular cities," we can use population to solve the issue of duplicate city names. 

By sorting the map data from highest to lowest population before removing duplicates, we guarantee that the pipeline always keeps the largest city. This ensures the weather data matches the correct major city, rather than a small town that just happens to share the same name.

In [60]:
# Create a lookup table from SimpleMaps
# Keep only most populous entry per city+country (avoid duplicates)
map_lookup = (
    simple_map_cities_df
    .sort_values('population', ascending=False)
    .drop_duplicates(subset=['city_clean', 'country_clean'], keep='first')
    .reset_index(drop=True)
)
map_lookup.head()

,city,city_ascii,lat,lng,country,iso2,iso3,region,capital,population,id,city_clean,region_clean,country_clean
0,Tokyo,Tokyo,35.6850,139.7514,Japan,JP,JPN,Tōkyō,primary,39105000.0,1.392686e+09,Tokyo,Tokyo,Japan
1,Jakarta,Jakarta,-6.1753,106.8269,Indonesia,ID,IDN,Jakarta,primary,33756000.0,1.360771e+09,Jakarta,Jakarta,Indonesia
2,Guangzhou,Guangzhou,23.1300,113.2600,China,CN,CHN,Guangdong,admin,26940000.0,1.156237e+09,Guangzhou,Guangdong,China
3,Mumbai,Mumbai,19.0758,72.8775,India,IN,IND,Mahārāshtra,admin,24973000.0,1.356227e+09,Mumbai,Maharashtra,India
4,Shanghai,Shanghai,31.2325,121.4692,China,CN,CHN,Shanghai,admin,24870895.0,1.156074e+09,Shanghai,Shanghai,China


In [61]:
# Tier1 merge - city + country only
tier1 = weather_df.merge(
    map_lookup[["city_clean", "country_clean", "region_clean", "lat", "lng", "population"]],
    on=["city_clean", "country_clean"],
    how="left",
    suffixes=('_weather', "_map")
)
tier1.head()

,city,region,country,month,high_temp_F,low_temp_F,mean_temp_F,precipitation_in,humidity_percent,dew_point_F,wind_mph,pressure_Hg,visibility_mi,city_clean,region_clean_weather,country_clean,region_clean_map,lat,lng,population
0,Accra,NaN,Ghana,January,89.0,75.0,82.0,1.57,75.0,73.0,14.0,29.85,5.0,Accra,NaN,Ghana,Greater Accra,5.556,-0.1969,1782150.0
1,Accra,NaN,Ghana,February,90.0,77.0,84.0,1.79,77.0,75.0,17.0,29.84,6.0,Accra,NaN,Ghana,Greater Accra,5.556,-0.1969,1782150.0
2,Accra,NaN,Ghana,March,91.0,78.0,84.0,2.29,78.0,76.0,17.0,29.83,7.0,Accra,NaN,Ghana,Greater Accra,5.556,-0.1969,1782150.0
3,Accra,NaN,Ghana,April,90.0,78.0,84.0,2.93,78.0,76.0,16.0,29.83,7.0,Accra,NaN,Ghana,Greater Accra,5.556,-0.1969,1782150.0
4,Accra,NaN,Ghana,May,89.0,77.0,83.0,5.11,80.0,75.0,15.0,29.87,8.0,Accra,NaN,Ghana,Greater Accra,5.556,-0.1969,1782150.0


In [62]:
# Check what matched
matched_t1 = tier1[tier1['lat'].notna()]
unmatched_t1 = tier1[tier1['lat'].isna()]

print(f"Tier 1 matched:   {len(matched_t1)}")
print(f"Tier 1 unmatched: {len(unmatched_t1)}")
print("\nStill unmatched cities:")
print(unmatched_t1[['city', 'region', 'country']].values)

Tier 1 matched:   1668
Tier 1 unmatched: 12

Still unmatched cities:
[['Kiritimati' 'Christmas Island' 'Kiribati']
 ['Kiritimati' 'Christmas Island' 'Kiribati']
 ['Kiritimati' 'Christmas Island' 'Kiribati']
 ['Kiritimati' 'Christmas Island' 'Kiribati']
 ['Kiritimati' 'Christmas Island' 'Kiribati']
 ['Kiritimati' 'Christmas Island' 'Kiribati']
 ['Kiritimati' 'Christmas Island' 'Kiribati']
 ['Kiritimati' 'Christmas Island' 'Kiribati']
 ['Kiritimati' 'Christmas Island' 'Kiribati']
 ['Kiritimati' 'Christmas Island' 'Kiribati']
 ['Kiritimati' 'Christmas Island' 'Kiribati']
 ['Kiritimati' 'Christmas Island' 'Kiribati']]


In [63]:
# Find cities that successfully matched
matched_t1 = tier1[tier1['lat'].notna()]

# Filter for rows where the weather dataset actually provided a region
has_region = matched_t1[matched_t1['region_clean_weather'].notna()]

# Find the rows where the weather region does NOT equal the map's region
region_mismatch = has_region[has_region['region_clean_weather'] != has_region['region_clean_map']]

# Drop duplicates, keeping only the first occurrence of each city/country combo
unique_mismatches = region_mismatch.drop_duplicates(subset=['city_clean', 'country_clean'])

# Print the comparison so you can see exactly what is different
print("Cities with conflicting regions:")
display(unique_mismatches[['city_clean', 'country_clean', 'region_clean_weather', 'region_clean_map']])

Cities with conflicting regions:


,city_clean,country_clean,region_clean_weather,region_clean_map
240,Paris,France,Paris,Ile-de-France
516,Jakarta,Indonesia,Jakarta Special Capital Region,Jakarta
576,Barcelona,Spain,Barcelona,Catalonia
612,Beijing,China,Beijing Municipality,Beijing
888,Shanghai,China,Shanghai Municipality,Shanghai
1164,London,United Kingdom,England,"London, City of"
1428,Vienna,Austria,Vienna,Wien
1620,Dubai,United Arab Emirates,Dubai,Dubayy


In [64]:
# Drop all the weather-based region columns
# (using errors='ignore' just in case one of them was already dropped)
tier1 = tier1.drop(columns=['region', 'region_clean', 'region_clean_weather'], errors='ignore')

# Rename the map's region column to be your official region column
tier1 = tier1.rename(columns={'region_clean_map': 'region'})

In [65]:
tier1.head()

,city,country,month,high_temp_F,low_temp_F,mean_temp_F,precipitation_in,humidity_percent,dew_point_F,wind_mph,pressure_Hg,visibility_mi,city_clean,country_clean,region,lat,lng,population
0,Accra,Ghana,January,89.0,75.0,82.0,1.57,75.0,73.0,14.0,29.85,5.0,Accra,Ghana,Greater Accra,5.556,-0.1969,1782150.0
1,Accra,Ghana,February,90.0,77.0,84.0,1.79,77.0,75.0,17.0,29.84,6.0,Accra,Ghana,Greater Accra,5.556,-0.1969,1782150.0
2,Accra,Ghana,March,91.0,78.0,84.0,2.29,78.0,76.0,17.0,29.83,7.0,Accra,Ghana,Greater Accra,5.556,-0.1969,1782150.0
3,Accra,Ghana,April,90.0,78.0,84.0,2.93,78.0,76.0,16.0,29.83,7.0,Accra,Ghana,Greater Accra,5.556,-0.1969,1782150.0
4,Accra,Ghana,May,89.0,77.0,83.0,5.11,80.0,75.0,15.0,29.87,8.0,Accra,Ghana,Greater Accra,5.556,-0.1969,1782150.0


In [66]:
tier1 = tier1[~((tier1['city'] == 'Kiritimati') & (tier1['country'] == 'Kiribati'))]
tier1 = tier1.reset_index(drop=True)
tier1.head()

,city,country,month,high_temp_F,low_temp_F,mean_temp_F,precipitation_in,humidity_percent,dew_point_F,wind_mph,pressure_Hg,visibility_mi,city_clean,country_clean,region,lat,lng,population
0,Accra,Ghana,January,89.0,75.0,82.0,1.57,75.0,73.0,14.0,29.85,5.0,Accra,Ghana,Greater Accra,5.556,-0.1969,1782150.0
1,Accra,Ghana,February,90.0,77.0,84.0,1.79,77.0,75.0,17.0,29.84,6.0,Accra,Ghana,Greater Accra,5.556,-0.1969,1782150.0
2,Accra,Ghana,March,91.0,78.0,84.0,2.29,78.0,76.0,17.0,29.83,7.0,Accra,Ghana,Greater Accra,5.556,-0.1969,1782150.0
3,Accra,Ghana,April,90.0,78.0,84.0,2.93,78.0,76.0,16.0,29.83,7.0,Accra,Ghana,Greater Accra,5.556,-0.1969,1782150.0
4,Accra,Ghana,May,89.0,77.0,83.0,5.11,80.0,75.0,15.0,29.87,8.0,Accra,Ghana,Greater Accra,5.556,-0.1969,1782150.0


In [67]:
tier1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1668 entries, 0 to 1667
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   city              1668 non-null   object 
 1   country           1668 non-null   object 
 2   month             1668 non-null   object 
 3   high_temp_F       1668 non-null   float64
 4   low_temp_F        1668 non-null   float64
 5   mean_temp_F       1668 non-null   float64
 6   precipitation_in  1668 non-null   float64
 7   humidity_percent  1668 non-null   float64
 8   dew_point_F       1668 non-null   float64
 9   wind_mph          1668 non-null   float64
 10  pressure_Hg       1668 non-null   float64
 11  visibility_mi     1656 non-null   float64
 12  city_clean        1668 non-null   object 
 13  country_clean     1668 non-null   object 
 14  region            1620 non-null   object 
 15  lat               1668 non-null   float64
 16  lng               1668 non-null   float64


## The Missing Data Check (Beyond the Merge Key)
We pulled in population from the map dataset. Verify that the cities that did merge actually had population data available

In [68]:
# Check for nulls in the newly attached map columns
print("\nMissing values in map columns:")
print(tier1[['lat', 'lng', 'population']].isna().sum())

# Find the rows where region is still NaN
missing_regions = tier1[tier1['region'].isna()]

# Isolate just the unique city and country names
cities_missing_region = missing_regions[['city_clean', 'country_clean']].drop_duplicates()

print(cities_missing_region.to_string(index=False))


Missing values in map columns:
lat           0
lng           0
population    0
dtype: int64
city_clean country_clean
    Nassau  Bahamas, The
 Hong Kong     Hong Kong
 Kathmandu         Nepal
 Singapore     Singapore


In [69]:
# Define the missing administrative regions
region_fixes = {
    'Nassau': 'New Providence',
    'Hong Kong': 'Hong Kong',
    'Kathmandu': 'Bagmati',
    'Singapore': 'Singapore'
}

# Apply the fixes to the region column
for city, region_name in region_fixes.items():
    tier1.loc[tier1['city_clean'] == city, 'region'] = region_name

# Final verification (should output 0)
print(f"Missing regions remaining: {tier1['region'].isna().sum()}")

Missing regions remaining: 0


### The Uniqueness Check
Verify that we still have exactly 12 distinct rows (months) for every single city/country combination, ensuring no data was dropped or duplicated at the granular level.

In [38]:
# Count how many records exist for each city/country pair
month_counts = tier1.groupby(['city_clean', 'country_clean']).size()

# Find any cities that don't have exactly 12 records
abnormal_cities = month_counts[month_counts != 12]

if abnormal_cities.empty:
    print("\nSuccess: All cities have exactly 12 months of data.")
else:
    print("\nWARNING: The following cities have missing or duplicate months:")
    print(abnormal_cities)


Success: All cities have exactly 12 months of data.


In [70]:
tier1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1668 entries, 0 to 1667
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   city              1668 non-null   object 
 1   country           1668 non-null   object 
 2   month             1668 non-null   object 
 3   high_temp_F       1668 non-null   float64
 4   low_temp_F        1668 non-null   float64
 5   mean_temp_F       1668 non-null   float64
 6   precipitation_in  1668 non-null   float64
 7   humidity_percent  1668 non-null   float64
 8   dew_point_F       1668 non-null   float64
 9   wind_mph          1668 non-null   float64
 10  pressure_Hg       1668 non-null   float64
 11  visibility_mi     1656 non-null   float64
 12  city_clean        1668 non-null   object 
 13  country_clean     1668 non-null   object 
 14  region            1668 non-null   object 
 15  lat               1668 non-null   float64
 16  lng               1668 non-null   float64


In [71]:
# Drop the original messy columns (errors='ignore' prevents crashes if 'city' is already gone)
tier1 = tier1.drop(columns=['country', 'city'], errors='ignore')

# Rename your clean columns to become the final official columns
tier1 = tier1.rename(columns={
    'city_clean': 'city',
    'country_clean': 'country'
})

# Check the final clean headers
tier1.head()

,month,high_temp_F,low_temp_F,mean_temp_F,precipitation_in,humidity_percent,dew_point_F,wind_mph,pressure_Hg,visibility_mi,city,country,region,lat,lng,population
0,January,89.0,75.0,82.0,1.57,75.0,73.0,14.0,29.85,5.0,Accra,Ghana,Greater Accra,5.556,-0.1969,1782150.0
1,February,90.0,77.0,84.0,1.79,77.0,75.0,17.0,29.84,6.0,Accra,Ghana,Greater Accra,5.556,-0.1969,1782150.0
2,March,91.0,78.0,84.0,2.29,78.0,76.0,17.0,29.83,7.0,Accra,Ghana,Greater Accra,5.556,-0.1969,1782150.0
3,April,90.0,78.0,84.0,2.93,78.0,76.0,16.0,29.83,7.0,Accra,Ghana,Greater Accra,5.556,-0.1969,1782150.0
4,May,89.0,77.0,83.0,5.11,80.0,75.0,15.0,29.87,8.0,Accra,Ghana,Greater Accra,5.556,-0.1969,1782150.0


In [72]:
# Save your final enriched dataset
tier1.to_csv("../climate_and_population_data.csv", index=False)

In [73]:
# Open connection to SQLite database
conn = sqlite3.connect("../db/city_climate_data.db")

# Save the final DataFrame to a SQLite database
tier1.to_sql("city_climate_data", conn, if_exists='replace', index=False)

# Quick verification query to confirm the write is succeeded
row_count = conn.execute("SELECT COUNT(*) FROM city_climate_data").fetchone()[0]
print(f"Successfully saved {row_count:,} rows to city_climate_data table.")

# Close the connection
conn.close()

Successfully saved 1,668 rows to city_climate_data table.
